<a href="https://colab.research.google.com/github/vlotran/HW2-Excellent-Bigframes-PyTorch/blob/main/notebooks/task2_bigframes_embeddings_retrieval_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2 — BigQuery Bigframes → BigQuery Text Embeddings → Neural Retrieval (Recall@K)

What this notebook does:

* Loads CFPB complaint narratives directly from BigQuery using Bigframes.
* Uses BigQuery’s text embedding model (text-embedding-005) to embed each complaint.
* Builds a semantic search / retrieval workflow: given one complaint, retrieve the most similar complaints using cosine similarity.
* Trains a small PyTorch neural projection head to improve retrieval, then evaluates with Recall@K before vs after the neural model.

## BigQuery source table:

* bigquery-public-data.cfpb_complaints.complaint_database

# 0. Setup (Auth + Project + Installs)

In [5]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "hw2-excellent"
REGION = "US"
!gcloud config set project {PROJECT_ID}

!pip -q install bigframes torch scikit-learn numpy

import bigframes.pandas as bf
bf.options.bigquery.project = PROJECT_ID
bf.options.bigquery.location = REGION

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


 # 1. Load data from BigQuery with Bigframes

In [6]:
import json

K = 8
N_PER_CLASS = 300

top_products = bf.read_gbq(f"""
WITH base AS (
  SELECT product, consumer_complaint_narrative AS text
  FROM `bigquery-public-data.cfpb_complaints.complaint_database`
  WHERE consumer_complaint_narrative IS NOT NULL
    AND product IS NOT NULL
)
SELECT product, COUNT(*) AS n
FROM base
GROUP BY product
HAVING n >= {N_PER_CLASS}
ORDER BY n DESC
LIMIT {K}
""").to_pandas()

products = top_products["product"].tolist()
products_sql = ", ".join(json.dumps(p) for p in products)

df = bf.read_gbq(f"""
WITH base AS (
  SELECT
    product,
    consumer_complaint_narrative AS text
  FROM `bigquery-public-data.cfpb_complaints.complaint_database`
  WHERE consumer_complaint_narrative IS NOT NULL
    AND product IN ({products_sql})
),
sampled AS (
  SELECT * FROM base
  QUALIFY ROW_NUMBER() OVER (PARTITION BY product ORDER BY RAND()) <= {N_PER_CLASS}
)
SELECT * FROM sampled
""")

df.head().to_pandas()

,product,text
0,Credit card or prepaid card,My wife and I have been purchasing from XXXX X...
1,Credit card or prepaid card,I am currently out of work and behind on bills...
2,Checking or savings account,This is my third complaint and everytime I get...
3,Checking or savings account,"On Thursday, XX/XX/2021, I attempted to close ..."
4,Student loan,Navient Refuses to lower my rates. XXXX XXXX h...


# 2. Generate embeddings in BigQuery

In [7]:
from bigframes.ml.llm import TextEmbeddingGenerator
import numpy as np

embedder = TextEmbeddingGenerator(model_name="text-embedding-005")
emb_df = embedder.predict(df[["text"]])

pdf = df.join(emb_df).to_pandas().dropna(subset=["ml_generate_embedding_result"]).reset_index(drop=True)
X = np.vstack(pdf["ml_generate_embedding_result"].apply(np.array).to_numpy()).astype(np.float32)
labels = pdf["product"].values

print("X:", X.shape, "labels:", len(labels))

/usr/local/lib/python3.12/dist-packages/bigframes/session/__init__.py:389: FutureWarning: You are using the BigFrames session default connection: bigframes-
default-connection,             which can be different from the
BigQuery project default connection.             This default
connection may change in the future.
  warnings.warn(msg, category=FutureWarning)


X: (2400, 768) labels: 2400


# 3. Retrieval evaluation baseline (cosine similarity) — Recall@K

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def recall_at_k(X, labels, k=10):
    sims = cosine_similarity(X, X)
    np.fill_diagonal(sims, -1.0)  # don't retrieve itself

    hits = 0
    for i in range(len(labels)):
        topk = np.argsort(-sims[i])[:k]
        if any(labels[j] == labels[i] for j in topk):
            hits += 1
    return hits / len(labels)

for k in [1, 3, 5, 10]:
    print(f"Recall@{k}: {recall_at_k(X, labels, k=k):.4f}")

Recall@1: 0.7275
Recall@3: 0.8879
Recall@5: 0.9271
Recall@10: 0.9637


# 4. Train a small neural projection head (metric learning-lite) and re-evaluate

In [9]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

D = X.shape[1]
proj = nn.Sequential(
    nn.Linear(D, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
).to(device)

# simple supervised objective: classify product from projected vector
# (this is not the same as Task 1 because our final metric is retrieval Recall@K)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)

clf = nn.Linear(128, len(le.classes_)).to(device)

opt = torch.optim.AdamW(list(proj.parameters()) + list(clf.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

Xt = torch.tensor(X, dtype=torch.float32).to(device)
yt = torch.tensor(y, dtype=torch.long).to(device)

for epoch in range(1, 11):
    proj.train(); clf.train()
    opt.zero_grad()
    z = proj(Xt)
    logits = clf(z)
    loss = criterion(logits, yt)
    loss.backward()
    opt.step()
    if epoch in [1, 3, 5, 10]:
        print("epoch", epoch, "loss", float(loss.detach().cpu()))

epoch 1 loss 2.0815134048461914
epoch 3 loss 2.06482195854187
epoch 5 loss 2.0418341159820557
epoch 10 loss 1.9383236169815063


## 4.1 Compute Recall@K after neural projection

In [10]:
proj.eval()
with torch.no_grad():
    Z = proj(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

# L2 normalize before cosine sim (common in retrieval)
Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)

for k in [1, 3, 5, 10]:
    print(f"Recall@{k} after projection: {recall_at_k(Z, labels, k=k):.4f}")

Recall@1 after projection: 0.7217
Recall@3 after projection: 0.8775
Recall@5 after projection: 0.9225
Recall@10 after projection: 0.9583
